## Benchmarking with Public Datasets 

Run the genomic prediction workflow on the prepared AG2P-DISC genotype and phenotype CSVs for common bean, durum wheat, maize, rice, and soybean.  
This notebook uses the same `MODEL_FAMILIES` and `_fit_and_evaluate_target` function in `src/capstone_pipeline.py` as the sweetpotato genomic prediction workflow.

These are benchmarks of the configured pipelines, not equally optimized model families: Elastic Net and MLP retain at most 50 markers, MLP uses at most eight PCA components, and Ridge uses all markers. Counts in the result tables are retained labeled samples per trait, not total genotype rows.

Run the cells from top to bottom using the project Python environment.  
Prepare the benchmarking input data CSVs in `src/external_validation/prepare_ag2p_benchmark.py` before running this notebook. 


In [1]:
# Run this setup cell first; later cells reuse its paths, dataset list, and imports.
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

# Find the repository root whether Jupyter starts in the root or notebooks/.
ROOT = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / "src" / "capstone_pipeline.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Could not locate src/capstone_pipeline.py from the current directory.")

sys.path.insert(0, str(ROOT / "src"))
from capstone_pipeline import MODEL_FAMILIES, _fit_and_evaluate_target # Import model families and the evaluation function from the pipeline.

# Input CSVs are read only; generated comparison CSVs are written to OUT_DIR.
PROCESSED_DIR = ROOT / "data" / "benchmarks" / "learnable" / "processed"
OUT_DIR = ROOT / "outputs" / "benchmarks" / "learnable"
# To run a subset, keep only the dataset slugs whose prepared CSV pairs you have.
DATASETS = ["commonbean", "durum_wheat", "maize", "rice210", "soybean"]
METRICS = ["cv_r2", "cv_mae", "cv_rmse"]

print(f"Project root: {ROOT}")
print(f"Model families: {', '.join(MODEL_FAMILIES)}")

Project root: C:\Users\frank\OneDrive - Mississippi State University\MADS\Capstone Project\capstone\fwk_capstone_v1.0
Model families: elastic_net, ridge, mlp_baseline


### Check prepared datasets

This analysis uses the processed CSVs already in `data/benchmarks/learnable/processed/`.  
Confirm the expected files, sample counts, marker counts, and trait counts before starting model fits.  

Every phenotype column other than `clone_id` is treated as a target.  
If a file is missing, prepare that dataset in `prepare_ag2p_benchmark.py` or remove its slug from `DATASETS`

In [2]:

# Check input availability before starting the more expensive model-fitting cell.
dataset_info = []
for dataset in DATASETS: # Iterate over each dataset to check for the availability of genotype and phenotype files.
    genotype_path = PROCESSED_DIR / f"genotype_external_{dataset}.csv"
    phenotype_path = PROCESSED_DIR / f"phenotype_external_{dataset}.csv"
    missing = [str(path) for path in (genotype_path, phenotype_path) if not path.is_file()]
    if missing:
        raise FileNotFoundError("Missing prepared benchmark file(s):\n" + "\n".join(missing))

    genotype = pd.read_csv(genotype_path)
    phenotype = pd.read_csv(phenotype_path)
    if "clone_id" not in genotype.columns or "clone_id" not in phenotype.columns:
        raise ValueError(f"Both files for {dataset!r} must contain a clone_id column.")
    # Append dataset-level information including sample counts, overlapping samples, marker and trait counts.
    dataset_info.append({
        "dataset": dataset,
        "genotype_samples": len(genotype),
        "phenotype_samples": len(phenotype),
        "overlapping_samples": phenotype["clone_id"].isin(genotype["clone_id"]).sum(),
        "markers": len(genotype.columns) - 1,
        "traits": len(phenotype.columns) - 1,
    })
# Convert the collected dataset information into a DataFrame for display and summary.
dataset_info = pd.DataFrame(dataset_info)
print("Dataset information:")
display(dataset_info)
print(f"Total dataset-trait comparisons expected: {dataset_info['traits'].sum()}")

Dataset information:


,dataset,genotype_samples,phenotype_samples,overlapping_samples,markers,traits
0,commonbean,471,471,471,5820,4
1,durum_wheat,1006,1006,1006,44536,1
2,maize,3528,3528,3528,4612,6
3,rice210,210,210,210,1619,4
4,soybean,346,346,346,31260,2


Total dataset-trait comparisons expected: 17


## Run cross-validated model comparisons
Evaluate every phenotype column against all model families.  

*Wait for the final `Finished` message before running the summary cells.*  

In [3]:
# Run after the input checks; progress messages identify the current model fit.
OUT_DIR.mkdir(parents=True, exist_ok=True)
comparison_frames = []
# Iterate over each dataset to fit models for all target traits and record the results.
for dataset in DATASETS:
    genotype_path = PROCESSED_DIR / f"genotype_external_{dataset}.csv"
    phenotype_path = PROCESSED_DIR / f"phenotype_external_{dataset}.csv"
    genotype = pd.read_csv(genotype_path)
    phenotype = pd.read_csv(phenotype_path)
    marker_columns = [column for column in genotype.columns if column != "clone_id"]
    targets = [column for column in phenotype.columns if column != "clone_id"]
    # Only IDs shared by both tables can contribute to model evaluation.
    labeled = phenotype.merge(genotype, on="clone_id", how="inner").set_index("clone_id")
    if labeled.empty:
        raise ValueError(f"No overlapping clone_id values for {dataset!r}.")

    rows = []
    for target in targets:
        for model_family in MODEL_FAMILIES:
            print(f"Evaluating {dataset} / {target} / {model_family} ...", flush=True)
            # Use the shared engine rather than fitting preprocessing outside CV.
            outcome = _fit_and_evaluate_target(
                target, labeled[marker_columns], labeled[target], model_family
            )
            if outcome.get("skipped"):
                raise RuntimeError(
                    f"Evaluation skipped for {dataset}/{target}/{model_family}: "
                    f"{outcome.get('skip_reason')}"
                )
            rows.append({
                "dataset": dataset,
                "target": target,
                "model_family": model_family,
                "n_samples": outcome["n_labeled_samples"],
                "n_markers": outcome["n_markers_available"],
                "n_selected_markers": outcome["n_selected_markers"],
                "n_cv_selected_markers": outcome["n_cv_selected_markers"],
                "n_pca_components": outcome["n_pca_components"],
                "n_cv_pca_components": outcome["n_cv_pca_components"],
                "cv_folds": outcome["n_cv_folds"],
                "cv_r2": outcome["cv_r2"],
                "cv_mae": outcome["cv_mae"],
                "cv_rmse": outcome["cv_rmse"],
                "overfitting_risk": outcome["overfitting_risk"],
            })
    # Append the dataset-level results to the comparison frames for later aggregation.
    dataset_results = pd.DataFrame(rows)
    per_dataset_path = OUT_DIR / f"model_comparison_{dataset}.csv" # Path to save the per-dataset model comparison results.
    # Preserve each completed crop's results before moving to the next crop.
    dataset_results.to_csv(per_dataset_path, index=False)
    print(f"Saved {per_dataset_path}")
    comparison_frames.append(dataset_results)
# Aggregate all per-dataset results into a single DataFrame for overall analysis.
results = pd.concat(comparison_frames, ignore_index=True) 
# Check that every requested trait has a result from every model family.
comparison_count = results[["dataset", "target"]].drop_duplicates().shape[0] # Count the number of unique dataset-target combinations actually evaluated.
expected_comparisons = int(dataset_info["traits"].sum())
if comparison_count != expected_comparisons:
    raise AssertionError(
        f"Expected {expected_comparisons} dataset-trait comparisons, got {comparison_count}."
    )
if len(results) != comparison_count * len(MODEL_FAMILIES):
    raise AssertionError("Unexpected number of model result rows.")

print(f"Finished {comparison_count} dataset-trait comparisons across {len(MODEL_FAMILIES)} models.")

Evaluating commonbean / pheno / elastic_net ...


Evaluating commonbean / pheno / ridge ...


Evaluating commonbean / pheno / mlp_baseline ...


Evaluating commonbean / pheno1 / elastic_net ...


Evaluating commonbean / pheno1 / ridge ...


Evaluating commonbean / pheno1 / mlp_baseline ...


... (52 outputs left out)

Evaluating soybean / pheno2 / elastic_net ...


Evaluating soybean / pheno2 / ridge ...


Evaluating soybean / pheno2 / mlp_baseline ...


Saved C:\Users\frank\OneDrive - Mississippi State University\MADS\Capstone Project\capstone\fwk_capstone_v1.0\outputs\benchmarks\learnable\model_comparison_soybean.csv
Finished 17 dataset-trait comparisons across 3 models.


### Aggregate and summarize results

In [4]:
# Run only after all model comparisons finish; this uses results held in memory.
results["data_type"] = "external"
column_order = [
    "dataset", "data_type", "target", "model_family", "n_samples", "n_markers",
    "n_selected_markers", "n_cv_selected_markers", "n_pca_components", "n_cv_pca_components",
    "cv_folds", "cv_r2", "cv_mae", "cv_rmse", "overfitting_risk",
]
results = results[column_order].sort_values(
    ["dataset", "target", "model_family"]
).reset_index(drop=True)
combined_path = OUT_DIR / "model_comparison_all_datasets.csv" # Path to save the combined model comparison results across all datasets.
# Rerunning overwrites the combined CSV with the currently selected datasets.
results.to_csv(combined_path, index=False)
print(f"Saved {combined_path}")

# Each dataset-trait comparison receives equal weight, regardless of sample size.
overall_means = results.groupby("model_family")[METRICS].mean().sort_index() # Compute the overall unweighted mean of each metric for each model family.
print(f"\nOverall unweighted means across {comparison_count} dataset-trait comparisons evaluated:")
display(overall_means)


Saved C:\Users\frank\OneDrive - Mississippi State University\MADS\Capstone Project\capstone\fwk_capstone_v1.0\outputs\benchmarks\learnable\model_comparison_all_datasets.csv

Overall unweighted means across 17 dataset-trait comparisons evaluated:


,cv_r2,cv_mae,cv_rmse
model_family,,,
elastic_net,0.269056,16.957762,21.759012
mlp_baseline,0.126471,17.646821,23.090921
ridge,0.449409,15.263569,19.714215


In [5]:
# Compute and display crop-level means for each model family.
crop_means = results.groupby(["dataset", "model_family"])[METRICS].mean().reset_index()
print("Crop-level means (not equally weighted in the overall summary):")
display(crop_means.pivot(index="dataset", columns="model_family", values="cv_r2"))

Crop-level means (not equally weighted in the overall summary):


model_family,elastic_net,mlp_baseline,ridge
dataset,,,
commonbean,0.279500,0.192174,0.400838
durum_wheat,0.130781,0.044077,0.263190
maize,0.374688,0.370114,0.591698
rice210,0.205980,-0.215844,0.448552
soybean,0.126563,-0.010040,0.214507


## Best model by trait and metric

Count how often each model family wins across the trait-level comparisons.

In [6]:
# Compare models within the same trait, where error metrics share the same units.
# Win counts describe these runs; they are not tests of statistical significance.
winner_rows = []
for metric, higher_is_better in [("cv_r2", True), ("cv_mae", False), ("cv_rmse", False)]:
    ordered = results.sort_values(metric, ascending=not higher_is_better)
    # Each trait gets one winner; exact ties keep the first row after sorting.
    winners = ordered.drop_duplicates(["dataset", "target"], keep="first")
    counts = winners["model_family"].value_counts().reindex(MODEL_FAMILIES, fill_value=0) 
    for model_family, count in counts.items():
        winner_rows.append({"metric": metric, "model_family": model_family, "trait_wins": count}) 

winner_summary = pd.DataFrame(winner_rows)
display(winner_summary.pivot(index="model_family", columns="metric", values="trait_wins"))

metric,cv_mae,cv_r2,cv_rmse
model_family,,,
elastic_net,0,0,0
mlp_baseline,0,0,0
ridge,17,17,17
